# Hybrid Data Integration Layer
## EcoGuard-ML Core Wildlife Threat Intelligence Platform

**Prepared by:** Antigravity (Principal Data Engineer, Geospatial Data Scientist, & Machine Learning Architect)
**Prepared for:** Wildlife Conservation Command Center

---

### Executive Brief
This notebook implements the data integration pipeline that upgrades **EcoGuard-ML Core** from a purely synthetic dataset system to a **Hybrid Data Intelligence Platform**. It merges real-world environmental, weather, and geospatial features with our threat events database while preserving total backward compatibility with all existing modeling and explainability pipelines.

Specifically, we integrate:
1. **Open-Meteo Weather**: Daily max temperature, precipitation, and relative humidity.
2. **OpenStreetMap (OSM) Roads**: Density scores calculated near roads and access tracks.
3. **NASA SRTM Elevation**: Topographic slope calculations derived from elevation differences.
4. **GBIF Wildlife Occurrences**: Species occurrence density aggregations.
5. **WDPA Protected Areas**: Shortest geodesic distance to the nearest park border.

The output is saved as `data/processed/hybrid_dataset.csv`, with a report generated at `reports/hybrid_data_report.md`.


## SECTION 1: LOAD DATA

We load the synthetic poaching incidents master dataset along with the raw external datasets representing real-world environmental and spatial information.


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import cKDTree
import os

# Relative paths from notebooks directory
master_path = '../data/raw/master_dataset.csv'
weather_path = '../data/external/open_meteo_weather.csv'
roads_path = '../data/external/osm_roads.geojson'
elev_path = '../data/external/nasa_srtm_elevation.csv'
gbif_path = '../data/external/gbif_wildlife_occurrences.csv'
wdpa_path = '../data/external/wdpa_protected_areas.geojson'
output_path = '../data/processed/hybrid_dataset.csv'

# Ingest files
df_master = pd.read_csv(master_path)
df_weather = pd.read_csv(weather_path)
gdf_roads = gpd.read_file(roads_path)
df_elev = pd.read_csv(elev_path)
df_gbif = pd.read_csv(gbif_path)
gdf_wdpa = gpd.read_file(wdpa_path)

print("=== Datasets Loaded Successfully ===")
print(f"Master Dataset: {df_master.shape}")
print(f"Open-Meteo Weather: {df_weather.shape}")
print(f"OSM Roads: {len(gdf_roads)} geometries")
print(f"NASA SRTM Elevation: {df_elev.shape}")
print(f"GBIF Occurrences: {df_gbif.shape}")
print(f"WDPA Protected Areas: {len(gdf_wdpa)} geometries")


## SECTION 2: SCHEMA VALIDATION

We validate coordinate ranges, column formats, and check for coordinate anomalies to ensure GIS alignment. The coordinates must represent the Serengeti reserve area.


In [ ]:
print("=== Validating Master Coordinates ===")
lat_min, lat_max = df_master['latitude'].min(), df_master['latitude'].max()
lon_min, lon_max = df_master['longitude'].min(), df_master['longitude'].max()

print(f"Latitude range: [{lat_min:.4f}, {lat_max:.4f}]")
print(f"Longitude range: [{lon_min:.4f}, {lon_max:.4f}]")

# Assert standard coordinate bounds for East Africa / Serengeti
assert -5.0 <= lat_min <= 0.0, f"Latitude out of bounds: {lat_min}"
assert 34.0 <= lon_min <= 38.0, f"Longitude out of bounds: {lon_min}"
assert df_master['latitude'].isnull().sum() == 0, "Latitude contains nulls"
assert df_master['longitude'].isnull().sum() == 0, "Longitude contains nulls"
assert df_master['poaching_incident'].isnull().sum() == 0, "Target contains nulls"

print("Master dataset coordinate check: PASSED")


## SECTION 3: DATA CLEANING

We inspect missing value profiles for all ingested external datasets and resolve potential gaps to ensure smooth down-stream merging.


In [ ]:
print("=== External Datasets Quality & Missing Values Check ===")
print(f"Weather Nulls: {df_weather.isnull().sum().sum()}")
print(f"OSM Roads Invalid Geometries: {gdf_roads.geometry.is_valid.all() == False}")
print(f"Elevation Nulls: {df_elev.isnull().sum().sum()}")
print(f"GBIF Nulls: {df_gbif.isnull().sum().sum()}")
print(f"WDPA Invalid Geometries: {gdf_wdpa.geometry.is_valid.all() == False}")

# Fix any potentially invalid geometries by adding a zero buffer
if not gdf_roads.geometry.is_valid.all():
    print("Correcting invalid OSM road geometries...")
    gdf_roads['geometry'] = gdf_roads.geometry.buffer(0)
    
if not gdf_wdpa.geometry.is_valid.all():
    print("Correcting invalid WDPA protected areas geometries...")
    gdf_wdpa['geometry'] = gdf_wdpa.geometry.buffer(0)

# Handle elevation or weather missing values if any
df_elev['real_elevation'] = df_elev['real_elevation'].fillna(df_elev['real_elevation'].median())
df_elev['terrain_slope'] = df_elev['terrain_slope'].fillna(df_elev['terrain_slope'].median())

print("Cleaning & imputation checks complete.")


## SECTION 4: COORDINATE STANDARDIZATION & PROJECTION

We standardize all coordinates into WGS84 (`EPSG:4326`) and project to UTM Zone 36S (`EPSG:32736`) representing Tanzania, to enable precise geodesic distance calculations in meters.


In [ ]:
# Create GeoDataFrames in WGS84
gdf_master = gpd.GeoDataFrame(
    df_master, 
    geometry=[Point(xy) for xy in zip(df_master['longitude'], df_master['latitude'])], 
    crs="EPSG:4326"
)
gdf_gbif = gpd.GeoDataFrame(
    df_gbif, 
    geometry=[Point(xy) for xy in zip(df_gbif['longitude'], df_gbif['latitude'])], 
    crs="EPSG:4326"
)

# Project to UTM Zone 36S (in meters)
gdf_master_utm = gdf_master.to_crs(epsg=32736)
gdf_roads_utm = gdf_roads.to_crs(epsg=32736)
gdf_wdpa_utm = gdf_wdpa.to_crs(epsg=32736)
gdf_gbif_utm = gdf_gbif.to_crs(epsg=32736)

print("All geospatial datasets projected to EPSG:32736 (UTM 36S)")


## SECTION 5: GEOSPATIAL & TEMPORAL MERGES

We execute the integration pipeline. We construct chronological daily dates mapping month indices to days of the year, join weather records, interpolate SRTM grids, compute road proximity buffers, and check species occurrence density indexes.


In [ ]:
print("=== Step A: Mapping Weather Parameters by Temporal Join ===")
# Re-assign sequential dates for temporal joins matching month structures
days_in_month = {
    1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30,
    7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31
}
np.random.seed(42)
days_in_month = {
    1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30,
    7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31
}

df_by_month = []
for m in range(1, 13):
    df_m = df_master[df_master['month'] == m].copy()
    n_rec = len(df_m)
    max_d = days_in_month[m]
    day_vals = [int(i * max_d / n_rec) + 1 for i in range(n_rec)]
    day_vals = [min(d, max_d) for d in day_vals]
    np.random.shuffle(day_vals)  # break the hour correlation
    
    df_m['day_of_month'] = day_vals
    df_m['date'] = df_m.apply(lambda r: f"2025-{m:02d}-{int(r['day_of_month']):02d}", axis=1)
    df_by_month.append(df_m)
    
df_master_dates = pd.concat(df_by_month)
df_master_dates['date'] = df_master_dates['date'].astype(str)
df_master_merged = df_master_dates.merge(df_weather, on='date', how='left')
df_master_merged = df_master_merged.drop(columns=['day_of_month'])

print("=== Step B: Topographic Slopes & Elevation via KDTree ===")
elev_tree = cKDTree(df_elev[['latitude', 'longitude']].values)
dists, indices = elev_tree.query(df_master_merged[['latitude', 'longitude']].values)
df_master_merged['real_elevation'] = df_elev.loc[indices, 'real_elevation'].values
df_master_merged['terrain_slope'] = df_elev.loc[indices, 'terrain_slope'].values

print("=== Step C: Reserve Proximity (Distance to Nearest WDPA border) ===")
reserve_proximity = gdf_master_utm.geometry.apply(lambda p: gdf_wdpa_utm.distance(p).min())
df_master_merged['reserve_proximity'] = reserve_proximity.values

print("=== Step D: OSM Road Density Score ===")
road_dist = gdf_master_utm.geometry.apply(lambda p: gdf_roads_utm.distance(p).min())
df_master_merged['road_density'] = 1000.0 / (road_dist.values + 10.0)

print("=== Step E: GBIF Wildlife Species Density within 10km ===")
species_list = ["Elephant", "Rhino", "Lion", "Buffalo", "Zebra"]
df_master_merged['species_occurrence_density'] = 0.0

for sp in species_list:
    gdf_gbif_sp = gdf_gbif_utm[gdf_gbif_utm['species'] == sp]
    if len(gdf_gbif_sp) == 0:
        continue
    
    sp_mask = df_master_merged['species'] == sp
    df_master_sp = df_master_merged[sp_mask]
    if len(df_master_sp) == 0:
        continue
        
    gbif_coords = np.array([[g.x, g.y] for g in gdf_gbif_sp.geometry])
    master_coords = np.array([[g.x, g.y] for g in gdf_master_utm[sp_mask].geometry])
    
    gbif_tree = cKDTree(gbif_coords)
    indices_list = gbif_tree.query_ball_point(master_coords, r=10000.0)
    
    sp_densities = []
    gbif_density_vals = gdf_gbif_sp['occurrence_density'].values
    for idxs in indices_list:
        if len(idxs) > 0:
            sp_densities.append(gbif_density_vals[idxs].sum())
        else:
            sp_densities.append(0.0)
            
    df_master_merged.loc[sp_mask, 'species_occurrence_density'] = sp_densities

# Clean up temporary date column to retain original columns plus new features
df_master_final = df_master_merged.drop(columns=['date'])
print("Ingestion & Merge Pipeline successfully executed.")


## SECTION 6: FEATURE EXTRACTION & ANALYSIS

We inspect statistical summaries and correlations of the newly integrated real-world features relative to the poaching target label.


In [ ]:
new_features = [
    'real_temperature', 'real_rainfall', 'real_humidity', 
    'real_elevation', 'terrain_slope', 'reserve_proximity', 
    'road_density', 'species_occurrence_density'
]

print("=== Statistics of New Features ===")
print(df_master_final[new_features].describe().T)

print("\n=== Correlation with poaching_incident target ===")
correlations = df_master_final[new_features + ['poaching_incident']].corr()['poaching_incident']
print(correlations.sort_values(ascending=False))


## SECTION 7: DATASET COMPILATION

We save the compiled hybrid dataset to its processed data output path.


In [ ]:
output_dir = os.path.dirname(output_path)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    
df_master_final.to_csv(output_path, index=False)
print(f"Unified hybrid dataset written to: {output_path}")
print(f"Final Dataset Shape: {df_master_final.shape}")


## SECTION 8: PIPELINE VERIFICATION

We verify that the target labels (`poaching_incident` == 1 counts) and core shapes match perfectly with the original synthetic data, assuring downstream notebook compatibility.
Specifically, the incident rate and total logs must remain identical to the master dataset.


In [ ]:
print("=== Validation Assertions ===")
print(f"Original incident count: {df_master['poaching_incident'].sum()}")
print(f"Hybrid incident count: {df_master_final['poaching_incident'].sum()}")

assert len(df_master) == len(df_master_final), f"Length mismatch: {len(df_master)} vs {len(df_master_final)}"
assert df_master['poaching_incident'].sum() == df_master_final['poaching_incident'].sum(), \
    "Poaching target sums mismatch!"
    
# Check for nulls in new columns
for col in new_features:
    assert df_master_final[col].isnull().sum() == 0, f"Null values detected in hybrid feature: {col}"
    
print("All backward compatibility validation checks: PASSED!")
